<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/moe_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


NUM_EXPERTS = 4
TOP_K = 2
HIDDEN = 8
NUM_TOKENS = 6

class Expert(nn.Module):
  def __init__(self, hidden):
    super().__init__()
    self.ffn = nn.Sequential(
        nn.Linear(hidden, hidden * 4),
        nn.ReLU(),
        nn.Linear(hidden * 4, hidden)
    )

  def forward(self, x):
    return self.ffn(x)


experts = nn.ModuleList([
    Expert(HIDDEN) for _ in range(NUM_EXPERTS)
])

router = nn.Linear(HIDDEN, NUM_EXPERTS)

x = torch.randn(NUM_TOKENS, HIDDEN)


router_logits = router(x)
router_probs = F.softmax(router_logits, dim=-1)


topk_probs, topk_experts = torch.topk(
    router_probs,
    k=TOP_K,
    dim=-1)

print("Selected experts per token:\n")


for token_id in range(NUM_TOKENS):
  print(
      f"Token {token_id} -> "
      f"Experts {topk_experts[token_id].tolist()}"
  )


print("\nExpert loads:\n")

for expert_id in range(NUM_EXPERTS):
  token_idx, slot_idx = torch.where(
      topk_experts == expert_id
  )

  print(
      f"Expert {expert_id}: "
      f"{len(token_idx)} token assignments"
  )


output = torch.zeros_like(x)

for expert_id in range(NUM_EXPERTS):

  token_idx, slot_idx = torch.where(
      topk_experts == expert_id
  )

  if len(token_idx) == 0:
    continue


  expert_input = x[token_idx]

  expert_output = experts[expert_id](
      expert_input
  )

  routing_weights = topk_probs[
      token_idx,
      slot_idx
  ]

  output[token_idx] += (
      expert_output *
      routing_weights.unsqueeze(-1)
  )


  print("\nOutput shape:", output.shape)

Selected experts per token:

Token 0 -> Experts [1, 0]
Token 1 -> Experts [0, 3]
Token 2 -> Experts [2, 0]
Token 3 -> Experts [1, 3]
Token 4 -> Experts [2, 0]
Token 5 -> Experts [2, 0]

Expert loads:

Expert 0: 5 token assignments
Expert 1: 2 token assignments
Expert 2: 3 token assignments
Expert 3: 2 token assignments

Output shape: torch.Size([6, 8])

Output shape: torch.Size([6, 8])

Output shape: torch.Size([6, 8])

Output shape: torch.Size([6, 8])


mock expert parallel

In [ ]:
expert_to_gpu = {
    0: "GPU 0",
    1: "GPU 0",
    2: "GPU 1",
    3: "GPU 1",
}

print("\nExpert placement:\n")

for expert_id, gpu in expert_to_gpu.items():
  print(f"Expert {expert_id} -> {gpu}")


Expert placement:

Expert 0 -> GPU 0
Expert 1 -> GPU 0
Expert 2 -> GPU 1
Expert 3 -> GPU 1


In [ ]:
print("\nToken destinations:\n")

for token_id in range(NUM_TOKENS):

  selected = topk_experts[token_id].tolist()

  destinations = [
      expert_to_gpu[e]
      for e in selected
  ]


  print(
      f"Token {token_id}: "
      f"{selected} -> {destinations}"
  )


Token destinations:

Token 0: [1, 0] -> ['GPU 0', 'GPU 0']
Token 1: [0, 3] -> ['GPU 0', 'GPU 1']
Token 2: [2, 0] -> ['GPU 1', 'GPU 0']
Token 3: [1, 3] -> ['GPU 0', 'GPU 1']
Token 4: [2, 0] -> ['GPU 1', 'GPU 0']
Token 5: [2, 0] -> ['GPU 1', 'GPU 0']
